In [34]:
import duckdb, httpx
import os, random, json, time
from rich import print
from rich.progress import track
import bikidata
import pandas as pd

# Demonstrating non-trival dataset sizes using DBLP

To compare the performance of bikiDATA with a current state-of-the-art triplestore, we index a non-trivial dataset with both and run some queries. We use the DBLP dataset as mentioned in the "Sparqloscope: A Generic Benchmark for the Comprehensive and Concise Performance Evaluation of SPARQL Engines" paper, [downloaded from here](https://drops.dagstuhl.de/entities/artifact/10.4230/dblp.rdf.ntriples.2025-04-01).

In [ ]:
!wget https://drops.dagstuhl.de/storage/artifacts/dblp/rdf/2025/dblp-2025-04-01.nt.gz

In [11]:
bikidata.build(["dblp-2025-04-01.nt.gz"])

DEBUG     bikidata 2026-03-05 13:22:18 Building Bikidata index with ['dblp-2025-04-01.nt.gz']
DEBUG     bikidata 2026-03-05 13:22:18 Good, there are no triples in bikidate table yet
INFO      bikidata 2026-03-05 14:10:31 No BIKIDATA_FTS_SETTINGS found, using default settings: ignore = '[^a-zA-Z0-9]+', strip_accents = 1, lower=1, stemmer='porter'


We perform fulltext queries, or select some random items to view.

In [27]:
r = bikidata.query({'filters':[{'p':'fts', 'o':'Knuth'}], 'size':4})
print(r['total'], "in total")
for x in r['results'].values():
    print(x)

877 in total

{
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/k/DonaldEKnuth>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/books/daglib/0000774>'],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Donald E. Knuth"'],
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    'id': '_:Sig_2254194596d5922a7bc4286d10fd6d2b_1',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/journals/corr/abs-2005-05421>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Craig Knuth"'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/264/9782>'],
    'id': '_:Sig_40d72ef4efef14d4a35b274f34baadd5_1',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/57/8029>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/conf/otm/KnuthHS16>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Magnus Knuth"'],
    'id': '_:Sig_5e204a1e7ba56dc9abca0d4caf5cf94a_1',
    'graph': []
}

{
    '<http://purl.org/spar/datacite/hasIdentifier>': ['_:ID_61bf66ed46f7c0685f8b4b111a117e69'],
    '<https://dblp.org/rdf/schema#primaryCreatorName>': ['"Eric J. Knuth"'],
    '<https://dblp.org/rdf/schema#creatorName>': ['"Eric J. Knuth"'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': ['"Eric J. Knuth"'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Creator>',
        '<https://dblp.org/rdf/schema#Person>'
    ],
    'id': '<https://dblp.org/pid/220/7978>',
    'graph': []
}

In [30]:
r = bikidata.query({'filters':[{'p':'id', 'o':'random 3'}]})
for x in r['results'].values():
    print(x)

{
    '<http://purl.org/spar/datacite/hasIdentifier>': [
        '_:ID_e65e9446514fc299784f81f1ffb82d1a',
        '_:ID_cdcea05162dfd196dbfb72ae449cd37d'
    ],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Reference>',
        '<https://dblp.org/rdf/schema#Publication>'
    ],
    '<http://www.w3.org/2002/07/owl#sameAs>': [
        '<https://doi.org/10.1007/978-0-387-31439-6_100227>',
        '<http://dx.doi.org/10.1007/978-0-387-31439-6_100227>'
    ],
    '<https://dblp.org/rdf/schema#pagination>': ['"691"'],
    '<https://dblp.org/rdf/schema#doi>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<https://dblp.org/rdf/schema#documentPage>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<https://dblp.org/rdf/schema#listedOnTocPage>': ['<https://dblp.org/db/reference/vision/vision2014>'],
    '<https://dblp.org/rdf/schema#title>': ['"Robust Clustering."'],
    '<https://dblp.org/rdf/schema#bibtexType>': ['<http://purl.org/net/nknouf/ns/bibtex#Incollection>'],
    '<https://dblp.org/rdf/schema#publishedIn>': ['"Computer Vision, A Reference Guide"'],
    '<https://dblp.org/rdf/schema#yearOfPublication>': ['"2014"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    '<https://dblp.org/rdf/schema#numberOfCreators>': ['"0"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#publishedInBook>': ['"Computer Vision, A Reference Guide"'],
    '<https://dblp.org/rdf/schema#primaryDocumentPage>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': ['"Robust Clustering. (2014)"'],
    'id': '<https://dblp.org/rec/reference/vision/X14hh>',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#documentPage>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#hasSignature>': ['_:Sig_91ab64695839d2f122d0927ba7714ef2_1'],
    '<http://www.w3.org/2002/07/owl#sameAs>': [
        '<http://dx.doi.org/10.1145/317825.317876>',
        '<https://doi.org/10.1145/317825.317876>'
    ],
    '<http://purl.org/spar/datacite/hasIdentifier>': [
        '_:ID_21dc5ec12519862a59fb2b96824f6677',
        '_:ID_24fc6b7ec9f8ebadb6219991193067ae'
    ],
    '<https://dblp.org/rdf/schema#bibtexType>': ['<http://purl.org/net/nknouf/ns/bibtex#Inproceedings>'],
    '<https://dblp.org/rdf/schema#authoredBy>': ['<https://dblp.org/pid/60/2881>'],
    '<https://dblp.org/rdf/schema#publishedIn>': ['"DAC"'],
    '<https://dblp.org/rdf/schema#pagination>': ['"312-318"'],
    '<https://dblp.org/rdf/schema#yearOfEvent>': ['"1985"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': [
        '"Thomas R. Smith: A data architecture for an uncertain design and manufacturing environment. (1985)"'
    ],
    '<https://dblp.org/rdf/schema#createdBy>': ['<https://dblp.org/pid/60/2881>'],
    '<https://dblp.org/rdf/schema#title>': [
        '"A data architecture for an uncertain design and manufacturing environment."'
    ],
    '<https://dblp.org/rdf/schema#numberOfCreators>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#primaryDocumentPage>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#publishedAsPartOf>': ['<https://dblp.org/rec/conf/dac/1985>'],
    '<https://dblp.org/rdf/schema#publishedInStream>': ['<https://dblp.org/streams/conf/dac>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Inproceedings>',
        '<https://dblp.org/rdf/schema#Publication>'
    ],
    '<https://dblp.org/rdf/schema#doi>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#listedOnTocPage>': ['<https://dblp.org/db/conf/dac/dac1985>'],
    '<https://dblp.org/rdf/schema#publishedInBook>': ['"DAC"'],
    '<https://dblp.org/rdf/schema#yearOfPublication>': ['"1985"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    'id': '<https://dblp.org/rec/conf/dac/Smith85>',
    'graph': []
}

{
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Richard Berlin"'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/83/6265>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/conf/dac/WinslettBPW85>'],
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"2"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    'id': '_:Sig_c8de18318571c9675acf05dd33da33e8_2',
    'graph': []
}

# Comparison with Sparqloscope queries

To do an evaluation of the performance, we compare some SPARQL queries from the Sparqloscope paper. We selected queries that had a runtime of more than a few seconds in the published results. The same query was then converted into SQL that can be run on the bikidata engine. A Qlever triplestore database was run on the same machine on which the bikidata DuckDB queries were run.

In [58]:
queries = {}
for i in range(1,15,1):
    queries[i] = {'id':i, 'sparql':open(f'{i}.sparql').read(), 'sql':open(f'{i}.sql').read() }

In [59]:
for i, q in queries.items():
    print(f'[bold green]{i} {'#'*80} [/bold green]')
    print(q['sparql'])
    print('-'*80)
    print(q['sql'])

1 ################################################################################ 

PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX dblps: <https://dblp.org/rdf/schema-2020-07-01#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(?s) AS ?count)
WHERE {
  ?s ?p ?o .
  FILTER isBlank(?s)
}

--------------------------------------------------------------------------------

select count(s) from triples join iris on s = hash where value like '_%'

2 ################################################################################ 

...

SELECT SUM(CASE WHEN STARTS_WITH(lit.value, '"a') THEN 1 ELSE 0 END) AS count
FROM triples AS t1
JOIN iris AS i1 ON i1.hash = t1.p AND i1.value = '<http://www.w3.org/2000/01/rdf-schema#label>'
JOIN literals AS lit ON lit.hash = t1.o

14 ################################################################################ 

PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> 

SELECT (SUM(xsd:integer(STRENDS(?o, "a"))) AS ?count) { ?s rdfs:label ?o }

--------------------------------------------------------------------------------

SELECT SUM(CASE WHEN ENDS_WITH(lit.value, 'a"') THEN 1 ELSE 0 END) AS count
FROM triples AS t1
JOIN iris AS i1 ON i1.hash = t1.p AND i1.value = '<http://www.w3.org/2000/01/rdf-schema#label>'
JOIN literals AS lit ON lit.hash = t1.o

In [72]:
endpoint = "http://localhost:60600/" # In this case we were running QLever on port 60600
for i, query in track(queries.items()):
    try:
        start = time.time()        
        response = httpx.post(
          endpoint,
          timeout=360,
          data={"query": query['sparql']},
          headers={"Accept": "application/sparql-results+json"}
        )
        end = time.time()
        query['sparql_result_raw'] = response.json() 
        query['sparql_duration'] = end-start
    except Exception as e:
        query['sparql_error'] = str(e)
print("Done!")    

Output()

Done!

In [ ]:
for q in queries.values():
    qq = q['sparql_result_raw'].get('results', {}).get('bindings', ["??"])[0]
    q['sparql_result'] = list(qq.values())[0]['value']


In [69]:
for i, query in track(queries.items()):
    try:
        if 'sql_error' in query:
            del query['sql_error']
        start = time.time()
        result = bikidata.raw().execute(query['sql']).fetchall()
        query['sql_result'] = result[0][0]
        end = time.time()
        query['sql_duration'] = end-start
    except Exception as e:
        traceback.print_exc()
        query['sql_error'] = str(e)
print("Done!")

Output()

Done!

# Cold start QLever, running the queries first time

In [70]:
exclude = {"sparql", "sql", "sparql_result_raw"}
df = pd.DataFrame([{k: v for k, v in d.items() if k not in exclude} for d in queries.values()])
df

,id,sparql_duration,sparql_result,sql_result,sql_duration
0,1,5.465955,241373570,503926494,1.494365
1,2,6.487778,52069156,52069156,2.539832
2,3,11.875705,26034578,26034578,15.122698
3,4,74.653473,369364948,421434646,2.705613
4,5,9.840442,26034578,26034578,1.496913
5,6,0.979035,0,0,1.608105
6,7,18.252488,466960,466960,1.655320
7,8,18.927638,466960,466960,1.638383
8,9,19.039446,729715,729713,1.721160
9,10,18.168952,916588643,939793994,1.738672


We see that the bikiData query execution is consistently significantly faster than the equivalent SPARQL queries in all cases. If we however re-run the queries again, the performance radically improves as QLever caches have warmed up.

# Re-run SPARQL queries with warm cache

In [74]:
exclude = {"sparql", "sql", "sparql_result_raw"}
df = pd.DataFrame([{k: v for k, v in d.items() if k not in exclude} for d in queries.values()])
df

,id,sparql_duration,sparql_result,sql_result,sql_duration
0,1,0.029890,241373570,503926494,1.494365
1,2,0.045512,52069156,52069156,2.539832
2,3,0.029594,26034578,26034578,15.122698
3,4,0.026642,369364948,421434646,2.705613
4,5,0.027134,26034578,26034578,1.496913
5,6,0.972397,0,0,1.608105
6,7,0.025333,466960,466960,1.655320
7,8,0.026837,466960,466960,1.638383
8,9,0.026077,729715,729713,1.721160
9,10,0.025492,916588643,939793994,1.738672
